# Ling-3.0-tiny MLX 量化与部署调用

配套教程：`01-Ling-3.0-tiny MLX 量化与部署调用.md`

在 Apple Silicon Mac（验证机型 MacBook Air M4 16GB）上用 MLX 完成 Ling-3.0-tiny 的
4-bit 量化与推理部署。

## 1. 环境准备

> 本章只适用于 Apple Silicon Mac。

In [ ]:
%%bash
python -m pip install --upgrade pip
pip config set global.index-url https://pypi.tuna.tsinghua.edu.cn/simple
pip install "mlx-lm>=0.32.0"

## 2. 模型下载（BF16 约 15GB）

In [ ]:
%%bash
modelscope download --model inclusionAI/Ling-3.0-tiny --local_dir ~/models/Ling-3.0-tiny

## 3. 本地量化为 4-bit（约 4.8GB）

> ModelScope 的 int4 预量化是 vLLM 专用格式，MLX 用不了，必须从 BF16 本地量化。

In [ ]:
%%bash
python -m mlx_lm.convert --model ~/models/Ling-3.0-tiny \
    --q-bits 4 --mlx-path ~/models/Ling-3.0-tiny-4bit --trust-remote-code

> 若报 `StrictDataclassFieldValidationError: Field 'kda_lower_bound' expected float`，
> 把量化前后两个目录 `config.json` 里的 `-5` / `0` 改成 `-5.0` / `0.0`。

## 4. 命令行推理

In [ ]:
%%bash
python -m mlx_lm.generate --model ~/models/Ling-3.0-tiny-4bit \
    --prompt "介绍一下什么是机器学习" --max-tokens 256

## 5. OpenAI 兼容服务

> 16GB 内存建议加 `--prompt-cache-size 2` 开前缀缓存；
> 暂时不要开 `--kv-bits 8`（见 ml-explore/mlx-lm#2057）。

In [ ]:
%%bash
python -m mlx_lm.server --model ~/models/Ling-3.0-tiny-4bit --port 8080 &
SERVER_PID=$!
sleep 30
curl http://127.0.0.1:8080/v1/chat/completions \
  -H "Content-Type: application/json" \
  -d '{
    "model": "default",
    "messages": [{"role": "user", "content": "介绍一下什么是机器学习"}],
    "max_tokens": 256
  }'
kill $SERVER_PID 2>/dev/null || true